In [ ]:
import os
import time
import boto3
from datetime import datetime
from boto3.dynamodb.conditions import Key, Attr

In [ ]:
os.environ.setdefault("AWS_PROFILE", "lpascual")

In [ ]:
dynamo_client = boto3.client("dynamodb")

In [ ]:
dynamo_client.list_tables()

In [ ]:
tables = dynamo_client.list_tables()["TableNames"]

In [ ]:
"ecom_jobs" in tables

In [ ]:
if "ecom_jobs" not in tables:
    response = dynamo_client.create_table(
        AttributeDefinitions=[
            {
                "AttributeName": "PK",
                "AttributeType": "S"
            },
            {
                "AttributeName": "SK",
                "AttributeType": "S"
            }            
        ],
        TableName="ecom_jobs",
        KeySchema=[
            {
                "AttributeName": "PK",
                "KeyType": "HASH"
            },
            {
                "AttributeName": "SK",
                "KeyType": "RANGE"
            }            
        ],
        BillingMode="PAY_PER_REQUEST"
    )
    print(response)

In [ ]:
tables = dynamo_client.list_tables()["TableNames"]
"ecom_jobs" in tables

In [ ]:
dynamodb_resource = boto3.resource("dynamodb")
ecom_jobs_table = dynamodb_resource.Table("ecom_jobs")

In [ ]:
start_timestamp = datetime.now()
time.sleep(10)
end_timestamp = datetime.now()

In [ ]:
job_details = {
    "PK": "JOB#INGESTION",
    "SK": "META",
    "type": "job",
    "description": "Converts csv files to parquet format",
    "is_active": True
}

In [ ]:
job_run_details = {
    "PK": "JOB#INGESTION",
    "SK": f"RUN#{start_timestamp.isoformat()}",
    "type": "run",
    "start_timestamp" : start_timestamp.isoformat(),
    "end_timestamp" : end_timestamp.isoformat(),
    "run_details" : {
        "input_file_metadata": [{
            "file_name" : "2019-Oct-subset-10p-enriched.csv",
            "size" : 1200000000
        }],
        "job_runtime": int((end_timestamp - start_timestamp).total_seconds())
    },
    "status": "completed"
}

In [ ]:
response = ecom_jobs_table.put_item(
    Item=job_details
)
print(response)

In [ ]:
response = ecom_jobs_table.put_item(
    Item=job_run_details
)
print(response)

In [ ]:
response = ecom_jobs_table.query(
    KeyConditionExpression=Key("PK").eq("JOB#INGESTION") & Key("SK").begins_with("META")
)

items = response.get("Items", [])
print(items)

In [ ]:
response = ecom_jobs_table.query(
    KeyConditionExpression=Key("PK").eq("JOB#INGESTION") & Key("SK").begins_with("RUN#")
)

items = response.get("Items", [])
print(items)

### JOB#BRONZETRANSFORMATION

Run the following cells one after each other twice to generate two separate runs for the same job

In [ ]:
start_timestamp = datetime.now()
time.sleep(10)
end_timestamp = datetime.now()

In [ ]:
job_run_details = {
    "PK": "JOB#BRONZETRANSFORMATION",
    "SK": f"RUN#{start_timestamp.isoformat()}",
    "type": "run",
    "start_timestamp" : start_timestamp.isoformat(),
    "end_timestamp" : end_timestamp.isoformat(),
    "run_details" : {
        "input_file_metadata": [{
            "file_name" : "run-bronze_transform_sink-5-part-block-0-0-r-00000-snappy.parquet",
            "size" : 1200000000
        }],
        "job_runtime": int((end_timestamp - start_timestamp).total_seconds())
    },
    "status": "completed"
}

In [ ]:
response = ecom_jobs_table.put_item(
    Item=job_run_details
)
print(response)

In [ ]:
response = ecom_jobs_table.query(
    KeyConditionExpression=Key("PK").eq("JOB#BRONZETRANSFORMATION") & Key("SK").begins_with("RUN#")
)

items = response.get("Items", [])
print(items)

In [ ]:
for item in items:
    print(item)